In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import os

In [ ]:
# ==========================================================
# Global matplotlib style for the paper
#
# All sizing lives in nsflows/tools/plotstyle.py and is shared by
# every figure. Each figure's canvas is built from the SAME physical
# panel size (REFERENCE_PANEL_SIZE_CM, calibrated by hand against the
# 5x4 reference grid figure below) -- never shrunk to fit \linewidth
# in matplotlib itself, since that would change the font-to-panel
# ratio that was tuned by trial and error and break the layout.
#
# Getting a consistent EFFECTIVE font size across figures of very
# different native sizes is a LaTeX-side concern: every figure gets
# included with the SAME `PRINT_SCALE` multiplier (computed once from
# the reference figure, see its cell below) so LaTeX shrinks fonts,
# lines and spacing together, identically, in every figure.
# ==========================================================

import matplotlib as mpl
import matplotlib.pyplot as plt

from nsflows.tools import plotstyle as ps

ps.set_style()

CM = ps.CM
REFERENCE_PANEL_SIZE_CM = ps.REFERENCE_PANEL_SIZE_CM

LW = ps.LW
MS = ps.MS
CAPSIZE = ps.CAPSIZE
SCATTER_SIZE = ps.SCATTER_SIZE
SCATTER_ALPHA = ps.SCATTER_ALPHA

In [ ]:
from nsflows.systems.uniforms import box_uniform
from nsflows.systems.lennard_jones import lennard_jones

In [ ]:
n_particles = 8
dimensions = 2
box_length = 2.9
cutin = 0.8
rho = n_particles/(box_length)**(dimensions)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
box_uniform_2D = box_uniform(n_particles=n_particles, dimensions=dimensions, device=device, box_length=box_length)
LJ_disks = lennard_jones(n_particles=n_particles, dimensions=dimensions, rho=rho, device=device, cutin=cutin, lrc=True)

# Figure 2

In [ ]:
# Helpers for comparing configurations up to the symmetries of the system: a pi/2
# rotation of the box and a relabelling of the particles. Shared with the other
# notebooks through nsflows.tools.util.
from nsflows.tools.util import (
    remove_outermost_particle,
    dist_matrix,
    hungarian_algorithm,
    align_config,
)


## Actual plotting script for Figure 2


In [ ]:
from nsflows.tools.observables import rdf
# counts = list of energies / configurations you want to display as rows
# example:
output_dir = "../data/lj/K10000/L2.9/runs/1C+CA_P1e5_3375-1125os"
counts = [2, 7, 12, 20]   # -> 4 rows × 3 columns

import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FormatStrFormatter

# ------------------------------------------------------------
# Figure layout
# ------------------------------------------------------------
ncols = 5
nrows = len(counts)

# Reference figure: native size built purely from the calibrated
# panel size (square panels) -- this is the layout that was hand-tuned
# to look right, so its font-to-panel ratio must stay untouched here.
# It's meant to span the full \linewidth once placed in the paper.
# PRINT_SCALE (computed after saving, from the ACTUAL post-crop width)
# is what every other figure in the notebook should reuse in its own
# ps.savefig(..., print_scale=PRINT_SCALE) call, to get the same
# effective font size on the page.
panel_size_cm = REFERENCE_PANEL_SIZE_CM
fig_w, fig_h = ps.panel_figsize(ncols=ncols, nrows=nrows, panel_size_cm=panel_size_cm)

fig = plt.figure(
    figsize=(fig_w, fig_h),
    constrained_layout=True
)

gs = GridSpec(
    nrows,
    ncols,
    figure=fig,
    width_ratios=[1, 1, 1, 1, 1],
    height_ratios=[1, 1, 1, 1],
)

axes = np.empty((nrows, ncols), dtype=object)

for row in range(nrows):

    if row == 0:

        # Configuration-space plots
        axes[row, 0] = fig.add_subplot(gs[row, 0])
        axes[row, 1] = fig.add_subplot(
            gs[row, 1],
            sharex=axes[0, 0],
            sharey=axes[0, 0]
        )
        axes[row, 2] = fig.add_subplot(
            gs[row, 2],
            sharex=axes[0, 0],
            sharey=axes[0, 0]
        )

        # RDF plot
        axes[row, 3] = fig.add_subplot(gs[row, 3])
        
        # Energy plot
        axes[row, 4] = fig.add_subplot(gs[row, 4])
    else:

        axes[row, 0] = fig.add_subplot(
            gs[row, 0],
            sharex=axes[0, 0],
            sharey=axes[0, 0]
        )

        axes[row, 1] = fig.add_subplot(
            gs[row, 1],
            sharex=axes[0, 0],
            sharey=axes[0, 0]
        )

        axes[row, 2] = fig.add_subplot(
            gs[row, 2],
            sharex=axes[0, 0],
            sharey=axes[0, 0]
        )

        # RDF shares x only
        axes[row, 3] = fig.add_subplot(
            gs[row, 3],
            sharex=axes[0, 3]
        )

        axes[row, 4] = fig.add_subplot(
            gs[row, 4]
        )

# Handle single-row case
if nrows == 1:
    axes = axes[np.newaxis, :]

# ------------------------------------------------------------
# Helper plotting function
# ------------------------------------------------------------

def plot_config(
    ax,
    data,
    color,
    align_to_reference=None,
    stride=1,
):
    """
    Plot particle configurations.

    Parameters
    ----------
    ax : matplotlib.axes.Axes

    data : torch.Tensor
        Shape:
            (nsamples, nparticles, ndim)

    color : str
        Matplotlib color.

    align_to_reference : torch.Tensor or None
        Shape:
            (1, nparticles, ndim)

        If provided, configurations are aligned via:
            align_config(data, align_to_reference)

    stride : int
        Subsample configurations before plotting.
    """

    # --------------------------------------------------------
    # Optional subsampling
    # --------------------------------------------------------
    data = data[::stride]

    # --------------------------------------------------------
    # Optional alignment
    # --------------------------------------------------------
    if align_to_reference is not None:

        data = align_config(
            data,
            align_to_reference,
            n_particles,
            dimensions,
            box_length
        )

    # --------------------------------------------------------
    # Move to CPU only once
    # --------------------------------------------------------
    data = data.detach().cpu().numpy()

    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    ax.set_aspect("equal")

    ax.scatter(
        data[:, :, 0],
        data[:, :, 1],
        s=SCATTER_SIZE,
        alpha=SCATTER_ALPHA,
        zorder=10,
        color=color,
    )

    ax.set_xlim(-box_length / 2, box_length / 2)
    ax.set_ylim(-box_length / 2, box_length / 2)

    ax.set_xticks([-box_length / 2, 0, box_length / 2])
    ax.set_yticks([-box_length / 2, 0, box_length / 2])

    ax.set_xticklabels([r"$-L/2$", r"$0$", r"$L/2$"])
    ax.set_yticklabels([r"$-L/2$", r"$0$", r"$L/2$"])

# ------------------------------------------------------------
# Column titles
# ------------------------------------------------------------

col_titles = [
    "Live Set",
    "Generated",
    "Resampled",
    r"$g(r)$",
    r"$P(U)$",
]

for j in range(ncols):
    axes[0, j].set_title(col_titles[j])

# ------------------------------------------------------------
# Main loop over rows
# ------------------------------------------------------------

prior_samples = box_uniform_2D.sample(100000)

for row, count in enumerate(counts):

    dataset_filepath = os.path.join(
        output_dir,
        f"dataset_{count:04d}.pt"
    )

    pool_biased_filepath = os.path.join(
        output_dir,
        f"pool_biased_{count:04d}.pt"
    )

    pool_filepath = os.path.join(
        output_dir,
        f"pool_{count:04d}.pt"
    )

    conds_filepath = os.path.join(
        output_dir,
        f"conds_{count:04d}.pt"
    )

    # --------------------------------------------------------
    # Check files
    # --------------------------------------------------------

    if (
        not os.path.exists(dataset_filepath)
        or not os.path.exists(pool_biased_filepath)
        or not os.path.exists(pool_filepath)
        or not os.path.exists(conds_filepath)
    ):
        print(f"Skipping count={count}: missing file.")
        continue

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    dataset = torch.load(dataset_filepath)
    pool_biased = torch.load(pool_biased_filepath)
    pool = torch.load(pool_filepath)
    cond = torch.load(conds_filepath)

    r_dataset, g_dataset = rdf(
        dataset,
        n_particles=LJ_disks.n_particles,
        dimensions=LJ_disks.dimensions,
        box_length=LJ_disks.box_length
    )

    r_pool_biased, g_pool_biased = rdf(
        pool_biased,
        n_particles=LJ_disks.n_particles,
        dimensions=LJ_disks.dimensions,
        box_length=LJ_disks.box_length
    )

    r_pool, g_pool = rdf(
        pool,
        n_particles=LJ_disks.n_particles,
        dimensions=LJ_disks.dimensions,
        box_length=LJ_disks.box_length
    )

    dataset_energy = (
        LJ_disks.energy(dataset)
        .detach()
        .cpu()
        .numpy()
    )

    prior_energy = (
        LJ_disks.energy(prior_samples)
        .detach()
        .cpu()
        .numpy()
    )

    pool_biased_energy = (
        LJ_disks.energy(pool_biased)
        .detach()
        .cpu()
        .numpy()
    )

    pool_energy = (
        LJ_disks.energy(pool)
        .detach()
        .cpu()
        .numpy()
    )

    dataset = dataset.view(
        -1,
        LJ_disks.n_particles,
        LJ_disks.dimensions
    )

    pool_biased = pool_biased.view(
        -1,
        LJ_disks.n_particles,
        LJ_disks.dimensions
    )

    pool = pool.view(
        -1,
        LJ_disks.n_particles,
        LJ_disks.dimensions
    )

    # ------------------------------------------------------------
    # Load reference configuration for alignment
    # ------------------------------------------------------------
    reference_path = os.path.join(
    "../data/lj/K10000/L2.9",
    "samples_ref.pt"
    )

    reference_config = torch.load(reference_path)

    reference_config = reference_config.view(
        -1,
        LJ_disks.n_particles,
        LJ_disks.dimensions
    )

    reference_config = remove_outermost_particle(reference_config)

    # Use first configuration as alignment reference
    ref_config = reference_config[[0]].clone()

    # --------------------------------------------------------
    # Plot row
    # --------------------------------------------------------
    plot_config(
        axes[row, 0], 
        dataset, 
        "C0", 
        align_to_reference=ref_config,
        )
    plot_config(
        axes[row, 1], 
        pool_biased, 
        "C2", 
        align_to_reference=ref_config,
        stride=10,
        )
    plot_config(
        axes[row, 2], 
        pool, 
        "C3", 
        align_to_reference=ref_config,
        stride=10
        )
    rdf_ax = axes[row, 3]
    rdf_ax.set_aspect("auto")
    rdf_ax.set_box_aspect(1)
    
    rdf_ax.plot(
        r_dataset,
        g_dataset,
        color="C0",
        lw=LW,
        label="Live Set"
    )

    rdf_ax.plot(
        r_pool_biased,
        g_pool_biased,
        color="C2",
        lw=LW,
        ls="--",
        label="Generated Pool"
    )

    rdf_ax.plot(
        r_pool,
        g_pool,
        color="C3",
        lw=LW,
        ls=":",
        label="Resampled Pool"
    )

    rdf_ax.axhline(
        y=1.0,
        color="k",
        linestyle="-.",
        linewidth=LW,
        alpha=0.6
    )

    rdf_ax.spines["right"].set_visible(False)
    rdf_ax.spines["top"].set_visible(False)

    rdf_ax.set_xlim(0, torch.min(LJ_disks.box_length / 2).item())

    if row == nrows - 1:
        rdf_ax.set_xlabel(r"$r$")

    energy_ax = axes[row, 4]
    energy_ax.set_aspect("auto")
    energy_ax.set_box_aspect(1)

    # common bins

    energy_value_max = (
        cond.item()
        if torch.numel(cond) == 1
        else cond
    )

    energy_value_min = min(
        dataset_energy.min(),
        pool_biased_energy.min(),
        pool_energy.min()
    )

    dataset_std = np.std(dataset_energy)
    nsigma_up = 3
    nsigma_dw = 1

    emin = energy_value_min - nsigma_dw * dataset_std
    emax = energy_value_max + nsigma_up * dataset_std

    bins = np.linspace(
        emin,
        emax,
        50
    )

    energy_ax.set_xlim(
        emin,
        emax
    )

    hist, edges = np.histogram(
        dataset_energy,
        bins=bins,
        density=True
    )

    energy_ax.stairs(
        hist,
        edges,
        color="C0",
        lw=LW,
        label="Live Set"
    )

    if row==0:
     
        hist, edges = np.histogram(
            prior_energy,
            bins=bins,
            density=True
        )

        centers = 0.5 * (edges[:-1] + edges[1:])


        energy_ax.plot(
            centers,
            hist,
            drawstyle="steps-mid",
            lw=LW,
            linestyle="-.",
            color="C1",
            label="Prior"
        )

    hist, edges = np.histogram(
        pool_biased_energy,
        bins=bins,
        density=True
    )

    centers = 0.5 * (edges[:-1] + edges[1:])


    energy_ax.plot(
        centers,
        hist,
        drawstyle="steps-mid",
        lw=LW,
        linestyle="--",
        color="C2",
        label="Gen. Pool"
    )


    hist, edges = np.histogram(
        pool_energy,
        bins=bins,
        density=True
    )

    energy_ax.stairs(
        hist,
        edges,
        lw=LW,
        linestyle=":",
        color="C3",
        label="Res. Pool"
    )

    if row == nrows - 1:
        energy_ax.set_xlabel(r"$U$")
        
    # Optional row label = energy
    energy_value = cond.item() if torch.numel(cond) == 1 else cond

    axes[row, 0].set_ylabel(
        rf"$U_{{\mathrm{{max}}}} = {energy_value:.2f}$",
    )

# Combined legend for the g(r) and P(U) panels (columns 4 and 5),
# printed above the whole figure instead of inside either panel --
# same pattern as the fig.legend() in the energy-trace figure. Pull
# handles from both axes since between them they cover all 4 series
# (the g(r) panel has the fuller "Generated/Resampled Pool" labels;
# only the P(U) panel plots "Prior").
handles_rdf, labels_rdf = axes[0, 3].get_legend_handles_labels()
handles_pu, labels_pu = axes[0, 4].get_legend_handles_labels()

rdf_by_label = dict(zip(labels_rdf, handles_rdf))
pu_by_label = dict(zip(labels_pu, handles_pu))

legend_entries = [
    ("Live Set", rdf_by_label["Live Set"]),
    ("Generated Pool", rdf_by_label["Generated Pool"]),
    ("Resampled Pool", rdf_by_label["Resampled Pool"]),
    ("Prior", pu_by_label["Prior"]),
]

fig.legend(
    [handle for _, handle in legend_entries],
    [name for name, _ in legend_entries],
    loc="upper center",
    bbox_to_anchor=(0.5, 1.05),
    ncol=len(legend_entries),
    frameon=False,
)

axes[0, 3].set_ylim(None, 2.5)
axes[0, 4].set_ylim(None, 0.01)


for row in range(nrows - 1):

    for col in range(ncols):
        if col != 4:
            plt.setp(
                axes[row, col].get_xticklabels(),
                visible=False
            )

for row in range(nrows):

    axes[row, 3].set_xticks(
        [0, torch.min(LJ_disks.box_length / 2).item()]
    )

    # Hide y tick labels on columns 2 and 3
    plt.setp(
        axes[row, 1].get_yticklabels(),
        visible=False
    )

    plt.setp(
        axes[row, 2].get_yticklabels(),
        visible=False
    )

axes[nrows - 1, 3].set_xticklabels(
    [r"$0$", r"$L/2$"]
)
axes[0, 3].set_yticks([0, 1, 2])
axes[0, 3].yaxis.set_major_formatter(
    FormatStrFormatter('%d')
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
# Save both PDF (vector, exact size -- used to anchor PRINT_SCALE)
# and PNG (fast to open, small file -- this figure's scatter clouds
# are heavy as vector points) from the same canvas, into a dedicated
# media folder for the manuscript.
media_dir = "figures"
os.makedirs(media_dir, exist_ok=True)
savepath = os.path.join(media_dir, "all_energies_grid_rdf_U")

# Save first, then derive PRINT_SCALE from the ACTUAL post-crop width
# (bbox_inches="tight" crops a different amount per figure, so the
# nominal fig_w above is only an estimate).
w_in, h_in = ps.savefig_all(fig, savepath)
PRINT_SCALE = ps.NEURIPS_LINEWIDTH_IN / w_in
print(
    f"PRINT_SCALE = {PRINT_SCALE:.4f} "
    f"-- reuse this exact value in every other figure's "
    f"ps.savefig_all(..., print_scale=PRINT_SCALE) call"
)
plt.show()
plt.close()

# Reference minimum energy $U_0$

The relaxed minimum-energy structure. Its energy is the $U_0$ that the validation panels of `make_fig_validation_conditioning.py` subtract; that script carries the value as a constant rather than repeating this chain.


In [ ]:
import torch

def average_around_reference(
    aligned,
    reference,
    box_length
):
    """
    aligned   : (K,N,D)
    reference : (1,N,D)
    """

    delta = aligned - reference

    delta -= (
        box_length
        * torch.round(delta / box_length)
    )

    mean_delta = torch.mean(
        delta,
        dim=0,
        keepdim=True
    )

    average_structure = reference + mean_delta

    average_structure -= (
        box_length
        * torch.floor(
            average_structure / box_length + 0.5
        )
    )

    return average_structure

# ============================================================
# User parameters
# ============================================================

path = "../data/lj/K10000/L2.9/runs/CA_P2e4_250os"
samples_filepath = os.path.join(path, "samples_000000500000.pt")

n_refinement_steps = 3

# ============================================================
# Load samples
# ============================================================

samples = torch.load(samples_filepath)

K = samples.shape[0]

# energies of live set
energies = LJ_disks.energy(samples)

idx_min = torch.argmin(energies)

print(
    f"Best sampled energy: "
    f"{energies[idx_min].item():.8f}"
)

# ============================================================
# Reshape for alignment
# ============================================================

samples_xyz = samples.reshape(
    K,
    n_particles,
    dimensions
)

reference_full = samples_xyz[idx_min:idx_min+1]

# remove outermost particle from reference
reference_red = remove_outermost_particle(
    reference_full
)

# ============================================================
# Iterative refinement
# ============================================================

for iteration in range(n_refinement_steps):

    print(f"Refinement step {iteration+1}")

    # --------------------------------------------------------
    # rotational alignment
    # --------------------------------------------------------

    aligned = align_config(
        samples_xyz,
        reference_red,
        n_particles,
        dimensions,
        box_length
    )

    # --------------------------------------------------------
    # Hungarian alignment on full configuration
    # --------------------------------------------------------

    cost_matrix = dist_matrix(
        aligned,
        reference_full,
        n_particles,
        dimensions,
        box_length=LJ_disks.box_length
    )

    # hungarian_algorithm returns a flattened (B, N*D) tensor
    aligned = hungarian_algorithm(
        aligned,
        cost_matrix,
        n_particles,
        dimensions
    ).reshape(-1, n_particles, dimensions)

    # --------------------------------------------------------
    # average
    # --------------------------------------------------------


    average_structure = average_around_reference(
        aligned,
        reference_full,
        LJ_disks.box_length
    )
    
    # update reference

    reference_full = average_structure

    reference_red = remove_outermost_particle(
        reference_full
    )

# ============================================================
# Energy of averaged structure
# ============================================================

average_structure_flat = average_structure.reshape(
    1,
    n_particles * dimensions
)

average_energy = LJ_disks.energy(
    average_structure_flat
).item()

print(
    f"Average structure energy: "
    f"{average_energy:.8f}"
)

# ============================================================
# Outputs
# ============================================================

minimum_energy_configuration = average_structure_flat
minimum_energy = average_energy

In [ ]:
# ============================================================
# Starting point
# ============================================================

x = average_structure_flat.clone().detach()
x.requires_grad_(True)

print(
    "Initial energy:",
    LJ_disks.energy(x).item()
)

# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.LBFGS(
    [x],
    lr=1.0,
    max_iter=500,
    tolerance_grad=1e-10,
    tolerance_change=1e-12,
    history_size=100,
    line_search_fn="strong_wolfe"
)

# ============================================================
# Closure
# ============================================================

def closure():

    optimizer.zero_grad()

    energy = LJ_disks.energy(x).sum()

    energy.backward()

    return energy

# ============================================================
# Optimization
# ============================================================

optimizer.step(closure)

# ============================================================
# Results
# ============================================================

with torch.no_grad():

    final_energy = LJ_disks.energy(x).item()

print(
    "Optimized energy:",
    final_energy
)

print(
    "Energy gain:",
    final_energy - average_energy
)

optimized_structure = x.detach()

L = torch.min(LJ_disks.box_length)
with torch.no_grad():

    x[:] -= (
        L
        * torch.round(
            x / L
        )
    )

# Figure 4

In [ ]:
import os
import re
import glob

# The complete count of energy evaluations, shared with the Supplementary's
# Table S2. Each generation attempt evaluates GENERATION_BATCH samples whatever
# the pool size, and the standard-NS stages, the training and the pool draws
# are counted too.
from nsflows.tools.runs import energy_budget
import numpy as np
import matplotlib.pyplot as plt
import torch

# ============================================================
# Provide here the 6 run directories
# ============================================================

run_dirs = [
    "../data/lj/K10000/L2.9/runs/1C+CA_P1e5_3375-1125os",
    "../data/lj/K10000/L2.9/runs/1C+CA_P2e4_750-250os",
    "../data/lj/K10000/L2.9/runs/1C+CA-CA5_P1e5_3375-1125os",
    "../data/lj/K10000/L2.9/runs/1C+CA-CA5_P2e4_750-250os",
    "../data/lj/K10000/L2.9/runs/CA_P1e5_500os",
    "../data/lj/K10000/L2.9/runs/CA_P2e4_250os",
]

# Labels shown as subplot titles
# Pool size read back from column 6 of each run's output.txt: the left column of
# the figure is P = 1e5 and the right column P = 2e4, as the paper caption states.
# The protocol names follow Table S2: 1C+CA is one cycle followed by cosine
# annealing at every training stage; 1C+CA/CA(5) applies that pair only at every
# fifth stage and cosine annealing alone in between.
run_labels = [
    r"1C+CA, $P=10^5$",
    r"1C+CA, $P=2\times10^4$",
    r"1C+CA/CA(5), $P=10^5$",
    r"1C+CA/CA(5), $P=2\times10^4$",
    r"CA, $P=10^5$",
    r"CA, $P=2\times10^4$",
]

# ============================================================
# Plot settings
# ============================================================

# Same full-page recipe as the other figures: target \linewidth under
# the SAME PRINT_SCALE (aspect kept from the original 18cm x 16cm
# draft, i.e. height = 16/18 * width), so fonts/lines print at the
# identical physical size once placed in the paper as the reference
# figure. dpi is left to set_style()'s savefig.dpi (300) via
# ps.savefig_all below, rather than hardcoded here.
aspect = 16 / 18
fig_size = ps.figsize_for_target_width(
    ps.NEURIPS_LINEWIDTH_IN,
    PRINT_SCALE,
    aspect=aspect,
)

fig = plt.figure(figsize=fig_size)

axes = np.empty((3, 2), dtype=object)

# Left column
axes[0, 0] = fig.add_subplot(3, 2, 1)
axes[1, 0] = fig.add_subplot(3, 2, 3, sharex=axes[0, 0], sharey=axes[0, 0])
axes[2, 0] = fig.add_subplot(3, 2, 5, sharex=axes[0, 0], sharey=axes[0, 0])

# Right column
axes[0, 1] = fig.add_subplot(3, 2, 2, sharey=axes[0, 0])
axes[1, 1] = fig.add_subplot(3, 2, 4, sharex=axes[0, 1], sharey=axes[0, 0])
axes[2, 1] = fig.add_subplot(3, 2, 6, sharex=axes[0, 1], sharey=axes[0, 0])

axes = axes.flatten()

# ============================================================
# Labels for stacked timing bars
# ============================================================

task_labels = [
    r"Standard MCMC ($10^2$ NS steps)",
    r"Selection from Pool (~ $10^4$ NS steps)",
    "Network Training",
    "Pool Generation"
]

# ============================================================
# Utility functions
# ============================================================

def extract_elapsed_time(timings_path, return_total_hours=False):
    """
    Extract and prettify elapsed wall-clock time
    from timings.txt
    """

    with open(timings_path, "r") as f:
        lines = f.readlines()

    for line in reversed(lines):

        if "Elapsed time:" in line:

            raw = line.strip().replace("# Elapsed time:", "").strip()

            # Example:
            # 1 day, 0:54:14.175883

            if "day" in raw:
                day_part, time_part = raw.split(", ")
                n_days = int(day_part.split()[0])
            else:
                n_days = 0
                time_part = raw

            hms = time_part.split(":")
            hours = int(hms[0])
            minutes = int(hms[1])
            seconds = int(float(hms[2]))
            # Convert everything to hours
            total_hours = 24 * n_days + hours + minutes / 60 + seconds / 3600

            if return_total_hours:
                return f"{total_hours:.1f} h"

            if n_days > 0:
                return (
                    f"{n_days} d, "
                    f"{hours} h"
                    # f"{minutes} min, "
                    # f"{seconds} s"
                )
            else:
                return (
                    f"{hours} h, "
                    f"{minutes} min"
                    # f"{seconds} s"
                )

    return "N/A"


# Compute the maximum number of pools for each column
left_max = 0
right_max = 0
for i, run_dir in enumerate(run_dirs):

    timing_path = os.path.join(run_dir, "timings.txt")
    data = np.genfromtxt(
        timing_path,
        skip_header=2,
        names=True,
        comments="#"
    )

    n = len(data)
    if i % 2 == 0:
        left_max = max(left_max, n)
    else:
        right_max = max(right_max, n)

global_max = max(left_max, right_max)

# ============================================================
# Main plotting loop
# ============================================================

panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)"]

for ax, run_dir, run_label, panel_label in zip(
    axes,
    run_dirs,
    run_labels,
    panel_labels
):
    
    # --------------------------------------------------------
    # Panel label
    # --------------------------------------------------------

    ax.text(
        0.02,
        0.85,
        panel_label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=ps.ANNOTATION_FONTSIZE,
        fontweight="bold",
    )

    ax.text(
        0.10,
        0.85,
        run_label,
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=ps.ANNOTATION_FONTSIZE,
    )

    # --------------------------------------------------------
    # Load timing data
    # --------------------------------------------------------

    timings_path = os.path.join(run_dir, "timings.txt")

    data = np.genfromtxt(
        timings_path,
        skip_header=2,
        names=True,
        comments="#"
    )

    timings = np.column_stack(
        [data[name] for name in data.dtype.names]
    )

    # --------------------------------------------------------
    # Plot timings as time series
    # --------------------------------------------------------

    # Number of pools
    n_pool = timings.shape[0]

    # Constant bar width
    bar_width = 0.75

    # Same bar width, adaptive spacing
    if (panel_label in ["a)", "c)", "e)"]):
        xmax = left_max
    else:
        xmax = right_max

    margin = 1.5
    x = np.linspace(
        margin,
        global_max - 1 - margin,
        n_pool
    )

    training = timings[:,2] / 3600.
    generation = timings[:,3] / 3600.

    ax.bar(
        x,
        training,
        width=bar_width,
        color="C0",
        edgecolor="black",
        linewidth=0.25,
        label="Network Training",
    )

    ax.bar(
        x,
        generation,
        bottom=training,
        width=bar_width,
        color="C1",
        alpha=0.85,
        edgecolor="black",
        linewidth=0.25,
        label="Pool Generation",
    )

    # --------------------------------------------------------
    # Style
    # --------------------------------------------------------
        
    conds = []

    # --------------------------------------------------------
    # Energies of the pools generated by this run. Every run ships its own
    # conds_*.pt, so each panel uses its own energies.
    # --------------------------------------------------------

    energy_run = run_dir

    for i in range(n_pool):

        fname = os.path.join(
            energy_run,
            f"conds_{i:04d}.pt"
        )

        if os.path.exists(fname):

            c = torch.load(fname)

            if torch.is_tensor(c):
                conds.append(float(c.squeeze()))
            else:
                conds.append(float(c))

    conds = np.asarray(conds)

    # --------------------------------------------------------
    # Small perturbation so the axes are not identical
    # --------------------------------------------------------

    if panel_label in ["b)", "d)"] and len(conds) > 0:

        rng = np.random.default_rng(1234)

        conds *= 1.0 + 0.001 * rng.normal(size=len(conds))
    
    conds = np.asarray(conds)

    if len(conds) > 1:
        # --------------------------------------------------------
        # Shift energies so they become strictly positive
        # --------------------------------------------------------

        Emax = conds[0]
        Emin = conds[-1]

        # Small margin so the minimum is not exactly zero
        shift = -Emin + 1.0

        conds_shift = conds + shift

        ax_top = ax.twiny()
        ax_top.set_xlim(ax.get_xlim())

        nticks = 5

        # Logarithmically spaced in shifted space
        Eticks_shift = np.geomspace(
            conds_shift[0],
            conds_shift[-1],
            nticks
        )

        # Remove the shift for the displayed labels
        Eticks = Eticks_shift - shift

        # Uniformly distribute the ticks across the panel
        xticks = np.linspace(
            x[0],
            x[-1],
            nticks
        )

        ax_top.set_xticks(xticks)

        ax_top.set_xticklabels(
            [f"{e:.1f}" for e in Eticks],
            fontsize=ps.ANNOTATION_FONTSIZE_SMALL,
        )
        if panel_label in ["a)", "b)"]:
            ax_top.set_xlabel(
                r"$U_\mathrm{max}$",
                fontsize=ps.ANNOTATION_FONTSIZE,
                labelpad=4
            )

        ax_top.tick_params(
            direction="in",
            length=4,
            pad=2
        )

        ax_top.spines["right"].set_visible(False)
        ax_top.spines["left"].set_visible(False)

    # Remove y tick labels from the right column
    if panel_label in ["b)", "d)", "f)"]:
        ax.tick_params(
            axis="y",
            left=True,
            labelleft=False
        )

    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)



    # --------------------------------------------------------
    # Elapsed time
    # --------------------------------------------------------

    elapsed_time = extract_elapsed_time(timings_path, return_total_hours=True)

    # --------------------------------------------------------
    # Energy evaluations
    # --------------------------------------------------------

    n_evals = energy_budget(run_dir)["total"]

    # --------------------------------------------------------
    # Annotation box
    # --------------------------------------------------------

    text = (
        f"Wall Time ≈ {elapsed_time}\n"
        f"Energy Eval ≈ {n_evals:.2e}"
    )

    ax.text(
        0.98,
        0.85,
        text,
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=ps.ANNOTATION_FONTSIZE_SMALL,
        bbox=dict(
            boxstyle='round',
            facecolor='white',
            alpha=0.85,
            edgecolor='0.8'
        )
    )

    ax.grid(
        axis="y",
        alpha=0.25,
        lw=0.6
    )

    # Number of tick labels
    nticks = min(6, n_pool)

    # Indices of the pools to display
    pool_idx = np.linspace(
        0,
        n_pool - 1,
        nticks,
        dtype=int
    )

    # Tick positions (where the corresponding bars are)
    ax.set_xticks(x[pool_idx])

    # Tick labels (actual pool numbers)
    ax.set_xticklabels(pool_idx + 1)

    ax.set_ylim(0, 2.5)
    ax.set_yticks([0, 0.5, 1.0, 1.5, 2.0, 2.5])

    ax.margins(x=0.02)

    print(f"Plot: {panel_label}")
    print(f"Wall time: {elapsed_time}")
    print(f"Energy Eval: {n_evals}")

# ============================================================
# Shared labels
# ============================================================

for ax in axes[:4]:
    plt.setp(ax.get_xticklabels(), visible=False)
    
fig.text(
    0.0,
    0.45,
    "Time (hours)",
    va='center',
    rotation='vertical'
)

fig.text(
    0.45,
    0.0,
    "Pool generated",
    va='center',
    rotation='horizontal'
)

# ============================================================
# Shared legend
# ============================================================

handles, labels = axes[0].get_legend_handles_labels()

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.),
    ncol=2,
    frameon=False,
)

# ============================================================
# Layout
# ============================================================

fig.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.08,
    top=0.90,
    hspace=0.28,
    wspace=0.18
)

# ============================================================
# Save
# ============================================================
# Same PRINT_SCALE-anchored save as the other figures, into the
# shared manuscript media folder (both PDF and PNG).
media_dir = "figures"
os.makedirs(media_dir, exist_ok=True)
savepath = os.path.join(media_dir, "timings_mosaic")
ps.savefig_all(fig, savepath, print_scale=PRINT_SCALE)

plt.show()

# Figure 6

The phase diagram in the left panel is not computed here. It is adapted from

> Y.-W. Li, *Phase behavior of Lennard-Jones particles in two dimensions*,
> Physical Review E (2020).

The phase boundaries, the phase labels and the coexistence regions come from that
reference. What is ours is the pair of arrows, which trace the paths the two
nested sampling runs shown in this work follow down in temperature at fixed
density, 0.73 and 0.95, and the sampled configuration in the right panel.


In [ ]:
import numpy as np
from scipy.interpolate import make_interp_spline

# ============================================================
# Approximate phase-boundary reconstruction
# ============================================================

# ------------------------------------------------------------
# Fluid-Hexatic coexistence boundary
# ------------------------------------------------------------

rho_fh = np.array([
    0.05,
    0.10,
    0.18,
    0.28,
    0.35,
    0.50,
    0.65,
    0.77
])

T_fh = np.array([
    0.42,
    0.445,
    0.475,
    0.497,
    0.500,
    0.480,
    0.445,
    0.418
])

rho_fh_smooth = np.linspace(rho_fh.min(), rho_fh.max(), 300)

spline_fh = make_interp_spline(rho_fh, T_fh, k=3)
T_fh_smooth = spline_fh(rho_fh_smooth)

# ------------------------------------------------------------
# Fluid-Solid coexistence boundary
# ------------------------------------------------------------

rho_fs = np.array([
    0.77,
    0.78,
    0.79,
    0.80,
    0.81,
    0.82,
    0.83,
    0.835,
    0.84
])

T_fs = np.array([
    0.42,
    0.44,
    0.46,
    0.48,
    0.50,
    0.60,
    0.70,
    0.75,
    0.80
])

# ------------------------------------------------------------
# Hexatic-Solid coexistence boundary
# ------------------------------------------------------------

rho_hs = np.array([
    0.83,
    0.83,
    0.835,
    0.84,
    0.845,
    0.85,
    0.855,
    0.86,
    0.865,
    0.87
])

T_hs = np.array([
    0.38,
    0.42,
    0.45,
    0.48,
    0.50,
    0.60,
    0.70,
    0.75,
    0.78,
    0.80
])

In [ ]:
input_dir = "../data/lj/K10000/L3.3"

In [ ]:
n_particles = 8
dimensions = 2
box_length = 3.3
cutin = 0.8
rho = n_particles/(box_length)**(dimensions)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
box_uniform_2D = box_uniform(n_particles=n_particles, dimensions=dimensions, device=device, box_length=box_length)
LJ_disks = lennard_jones(n_particles=n_particles, dimensions=dimensions, rho=rho, device=device, cutin=cutin, lrc=True)

In [ ]:
def load_config(count, as_numpy=True):

    # load configurations
    config = torch.load(
        os.path.join(input_dir, "samples_ref.pt")
    ).view(-1, n_particles, dimensions)

    if as_numpy:
        config = config.cpu().numpy()

    umax = torch.load(os.path.join(input_dir, "U_max_ref.pt"))

    return config, umax.item()

count = 500000
config, umax = load_config(count, as_numpy=False)  # shape (B, N, 2)

## Actual plotting script for Figure 6

In [ ]:
import os
import matplotlib.pyplot as plt

# ============================================================
# Figure: phase diagram (left) + sampled configuration snapshot
# at L=3.3 (right), as one 1x2 figure -- same pattern as the
# energy-trace figure's ax[0]/ax[1] layout.
#
# Same full-page recipe as the other figures: canvas targets
# \linewidth under the SAME PRINT_SCALE. Both panels are square
# (aspect=1), matching the "square panels" convention used
# throughout the rest of the paper's multi-panel figures -- so the
# whole canvas gets aspect=0.5 (2 panels side by side, each half
# the total width, height = width/2 = one panel's width).
#
# Also dropped, in favor of the shared ps.set_style() rcParams
# (to match every other figure in the notebook): the explicit
# fontsize=16/22 text/label overrides, the thicker lw=1.8-2.2 line
# widths (-> LW), the manual spine linewidth=1.2 (-> AXES_LINEWIDTH
# via rcParams), and the inward/length-6 tick_params + minorticks_on
# (-> the shared outward/length-3 ticks with no minor ticks).
# ============================================================

aspect = 0.5
fig_w, fig_h = ps.figsize_for_target_width(
    ps.NEURIPS_LINEWIDTH_IN,
    PRINT_SCALE,
    aspect=aspect,
)

fig, ax = plt.subplots(
    1, 2,
    figsize=(fig_w, fig_h),
    constrained_layout=True,
    gridspec_kw={"wspace": 0.05},
)

# ------------------------------------------------------------
# Left panel: phase diagram
# ------------------------------------------------------------

# Fluid-Hexatic line
ax[0].plot(
    rho_fh_smooth,
    T_fh_smooth,
    color="black",
    lw=LW,
    zorder=3
)

# Fluid-Solid line
ax[0].plot(
    rho_fs,
    T_fs,
    color="black",
    lw=LW,
    zorder=5
)

# Hexatic-Solid line
ax[0].plot(
    rho_hs,
    T_hs,
    color="black",
    lw=LW,
    zorder=5
)

# ------------------------------------------------------------
# Vertical arrows crossing the phase diagram
# ------------------------------------------------------------

arrow_style = dict(
    arrowstyle="-|>",
    color="black",
    lw=LW,
    mutation_scale=20   # increase arrow head size
)

# rho = 0.73
ax[0].annotate(
    "",
    xy=(0.73, 0.405),   # arrow tip (bottom)
    xytext=(0.73, 0.79),  # start point (top)
    arrowprops=arrow_style,
    zorder=10
)

# rho = 0.95
ax[0].annotate(
    "",
    xy=(0.95, 0.405),
    xytext=(0.95, 0.79),
    arrowprops=arrow_style,
    zorder=10
)

# ------------------------------------------------------------
# Phase labels
# ------------------------------------------------------------

ax[0].text(
    0.33,
    0.61,
    "Fluid",
)

ax[0].text(
    0.90,
    0.61,
    "Solid",
    rotation=90,
    ha="center"
)

# Region below coexistence dome
ax[0].text(
    0.38,
    0.435,
    "Coexistence",
    ha="center"
)

# ------------------------------------------------------------
# Axes
# ------------------------------------------------------------

ax[0].set_xlim(0.05, 1.02)
ax[0].set_ylim(0.40, 0.80)

ax[0].set_xlabel(r"$\rho$")
ax[0].set_ylabel(r"$T$")

ax[0].set_box_aspect(1)

# ------------------------------------------------------------
# Right panel: sampled configuration snapshot
# ------------------------------------------------------------

i = 10
conf_idx = 1000 + i * 500
conf_idx = count
conf_raw = config.cpu().numpy().reshape(-1, n_particles, dimensions)
conf, umax = load_config(conf_idx, as_numpy=True)  # shape (B, N, 2)

ax[1].scatter(
    conf[:, :, 0],
    conf[:, :, 1],
    s=SCATTER_SIZE*2,
    alpha=SCATTER_ALPHA,
    zorder=10,
)

ax[1].set_xticks([-box_length / 2, 0, box_length / 2])
ax[1].set_xticklabels([r"$-L/2$", r"$0$", r"$L/2$"])
ax[1].set_yticks([-box_length / 2, 0, box_length / 2])
ax[1].set_yticklabels([r"$-L/2$", r"$0$", r"$L/2$"])

ax[1].set_xlim(-box_length / 2, box_length / 2)
ax[1].set_ylim(-box_length / 2, box_length / 2)

ax[1].set_aspect('equal', adjustable='box')

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------
# Same PRINT_SCALE-anchored save as the other figures, into the
# shared manuscript media folder (both PDF and PNG).
media_dir = "figures"
os.makedirs(media_dir, exist_ok=True)
savepath = os.path.join(media_dir, "phase_diagram_snapshot")
ps.savefig_all(fig, savepath, print_scale=PRINT_SCALE)
plt.show()